# 13. Despliegue de Modelos de Machine Learning y Deep Learning

Este notebook cubre el proceso de despliegue de modelos usando FastAPI, Docker y buenas prácticas para APIs de inferencia.

## Objetivo
- Aprender a serializar y cargar modelos entrenados.
- Crear una API REST de inferencia con FastAPI.
- Conocer las bases de contenedorización con Docker.
- Aplicar buenas prácticas de despliegue y versionamiento.

## Prerequisitos

> 📌 **Prerequisitos:** Haber completado al menos los notebooks [01](./01_intro_machine_learning.ipynb) y [03](./03_modelos_clasicos_ml.ipynb).

- Conceptos de entrenamiento y evaluación de modelos.

> ⚠️ **Dependencias adicionales:** `pip install fastapi uvicorn nest_asyncio`

## 1. Introducción teórica

El despliegue es el paso final del ciclo de vida de ML: llevar un modelo entrenado a producción para que pueda ser usado por aplicaciones reales.

**Etapas del despliegue:**
1. Serialización del modelo (guardar/cargar).
2. Creación de una API de inferencia.
3. Contenedorización (Docker).
4. Despliegue en la nube o servidor.

| Herramienta | Uso |
|-------------|-----|
| **joblib / pickle** | Serialización de modelos scikit-learn |
| **model.save()** | Serialización de modelos Keras |
| **FastAPI** | API REST de inferencia |
| **Docker** | Contenedorización para portabilidad |
| **MLflow / DVC** | Versionamiento de modelos y experimentos |

## 2. Entrenar y guardar un modelo

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

# Entrenar modelo de ejemplo
data = load_iris()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
print(f'Accuracy en test: {accuracy_score(y_test, clf.predict(X_test)):.2f}')

# Guardar modelo
joblib.dump(clf, 'modelo_iris.joblib')
print('Modelo guardado como modelo_iris.joblib')

## 3. Cargar el modelo guardado

In [ ]:
# Cargar modelo guardado
del clf
modelo = joblib.load('modelo_iris.joblib')

# Probar predicción
sample = X[0:1]
pred = modelo.predict(sample)
prob = modelo.predict_proba(sample)
print(f'Predicción: {data.target_names[pred[0]]} ({pred[0]})')
print(f'Probabilidades: {prob[0].round(3)}')

## 4. Crear y lanzar una API REST desde el notebook

Usaremos FastAPI y `nest_asyncio` para lanzar la API dentro del notebook.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np
import nest_asyncio
import uvicorn
from threading import Thread

# Cargar modelo
modelo = joblib.load('modelo_iris.joblib')

app = FastAPI(title='Iris Classifier API', version='1.0')

class InputData(BaseModel):
    features: list

class PredictionResponse(BaseModel):
    prediction: int
    class_name: str
    probabilities: list

@app.post('/predict', response_model=PredictionResponse)
def predict(data: InputData):
    X = np.array(data.features).reshape(1, -1)
    pred = modelo.predict(X)
    proba = modelo.predict_proba(X)[0].tolist()
    class_names = ['setosa', 'versicolor', 'virginica']
    return PredictionResponse(
        prediction=int(pred[0]),
        class_name=class_names[int(pred[0])],
        probabilities=proba
    )

@app.get('/health')
def health():
    return {'status': 'ok'}

# Permitir reinicio de bucle de eventos en Jupyter
nest_asyncio.apply()

def run_api():
    uvicorn.run(app, host='0.0.0.0', port=8000)

# Lanzar API en un hilo
api_thread = Thread(target=run_api, daemon=True)
api_thread.start()
print('API lanzada en http://localhost:8000')
print('Documentación automática en http://localhost:8000/docs')

## 5. Probar la API localmente

In [ ]:
import requests
import time

# Esperar a que la API esté lista
time.sleep(2)

# Health check
try:
    health = requests.get('http://127.0.0.1:8000/health')
    print('Health check:', health.json())
except:
    print('La API aún no está lista, intenta más tarde.')

# Predicción
url = 'http://127.0.0.1:8000/predict'
data = {"features": [5.1, 3.5, 1.4, 0.2]}
response = requests.post(url, json=data)
print('Respuesta de la API:', response.json())

## 6. Contenedorización con Docker

Para desplegar en cualquier servidor, creamos un contenedor Docker. A continuación, los archivos necesarios:

### `Dockerfile`

```dockerfile
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY modelo_iris.joblib .
COPY app.py .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```

### `requirements.txt`

```
fastapi
uvicorn
scikit-learn
numpy
joblib
```

### `docker-compose.yml`

```yaml
version: '3.8'
services:
  api:
    build: .
    ports:
      - "8000:8000"
    restart: unless-stopped
```

### Comandos para desplegar:

```bash
# Construir imagen
docker build -t iris-api .

# Ejecutar contenedor
docker run -p 8000:8000 iris-api

# Con docker-compose
docker-compose up -d
```

## 7. Versionamiento de modelos

En producción, es esencial versionar los modelos:

| Herramienta | Ventaja | Cuándo usar |
|-------------|---------|-------------|
| **MLflow** | Tracking de experimentos, registro de modelos | Equipos medianos/grandes |
| **DVC** | Control de versiones de datos y modelos | Integración con Git |
| **Convención manual** | Nombres con timestamp y métricas | Proyectos pequeños |

**Ejemplo de versionamiento manual:**

In [ ]:
from datetime import datetime
import json

# Guardar modelo con versión
version = datetime.now().strftime('%Y%m%d_%H%M%S')
model_path = f'modelo_iris_v{version}.joblib'
joblib.dump(modelo, model_path)

# Guardar metadata
metadata = {
    'version': version,
    'model_type': 'RandomForestClassifier',
    'accuracy': accuracy_score(y_test, modelo.predict(X_test)),
    'features': list(data.feature_names),
    'target_names': list(data.target_names)
}

with open(f'modelo_iris_v{version}_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Modelo guardado: {model_path}')
print(f'Metadata: {json.dumps(metadata, indent=2)}')

## 8. Discusión y Conclusiones

- Serializar modelos con `joblib` es simple y eficiente para scikit-learn.
- FastAPI permite crear APIs de inferencia rápidamente con documentación automática.
- Docker garantiza portabilidad y consistencia del entorno.
- El versionamiento de modelos es crítico para reproducibilidad en producción.
- Siempre incluir validación de datos, health checks, y manejo de errores.

## 9. Ejercicios Propuestos

1. **Ejercicio 1:** Modifica la API para que devuelva también las probabilidades por clase en formato de diccionario con los nombres de las clases.

2. **Ejercicio 2:** Agrega validación de entrada (que `features` tenga exactamente 4 elementos y sean numéricos).

3. **Ejercicio 3:** Crea los archivos `Dockerfile`, `requirements.txt` y `app.py` y construye la imagen Docker.

4. **Ejercicio 4 (Avanzado):** Configura MLflow para trackear experimentos y registra el modelo con diferentes hiperparámetros.

## 10. Referencias y Recursos

- [FastAPI Documentation](https://fastapi.tiangolo.com/)
- [scikit-learn: Model Persistence](https://scikit-learn.org/stable/model_persistence.html)
- [Docker Getting Started](https://docs.docker.com/get-started/)
- [MLflow Documentation](https://mlflow.org/docs/latest/index.html)

---

📎 **Notebook anterior:** [12. CPU, GPU y Metal](./12_cpu_gpu_metal.ipynb)  
📎 **Este es el último notebook del curso.** ¡Felicidades por completar el recorrido! 🎉